# 04 · English TTS benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/claverfred/real_time_swahili_english_interpreter/blob/main/notebooks/04_tts_benchmark.ipynb)

Compares English voices on first-audio latency, real-time factor, memory and an **intelligibility proxy**
(each clip is transcribed back with Whisper and scored by WER against the input text). Text: 25 FLORES-200
English sentences plus a fixed set of names, numbers and acronyms.

The proxy is not a listening test: naturalness needs human ratings (`docs/human_evaluation.md`).
Piper runs on the CPU through onnxruntime even on a GPU runtime; MMS uses PyTorch (GPU when available).
The CPU reference (development PC): Piper libritts_r first audio 0.22 s median, RTF 0.06, round-trip WER 0.074.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is enough; L4/A100/L40S are faster). Optional: add a Hugging Face token as the Colab Secret `HF_TOKEN` (key icon on the left); every default model is public, so it is not required.

Results, converted models and data are kept on Google Drive in `MyDrive/swahili_interpreter/`, so later sessions start faster. Nothing is uploaded anywhere else.

In [ ]:
# @title 1. Setup: clone/update the repository, mount Drive, install, fetch data (safe to re-run)
REPO_URL = "https://github.com/claverfred/real_time_swahili_english_interpreter.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
USE_DRIVE = True  # @param {type:"boolean"}
PROFILE = "colab"  # @param {type:"string"}

import os, subprocess, sys
WORKDIR = "/content" if os.path.isdir("/content") else os.getcwd()
REPO = os.path.join(WORKDIR, REPO_URL.rstrip("/").rsplit("/", 1)[-1].removesuffix(".git"))
if not os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--format=commit %h  %cd  %s"], capture_output=True, text=True).stdout)

from src.platform import colab
info = colab.bootstrap(REPO, use_drive=USE_DRIVE)
print("persistent root:", info["root"], "(Google Drive)" if info["on_drive"] else "")
for n in info["notes"]:
    print("NOTE:", n)
print("Hugging Face token:", "from Colab Secrets" if info["hf_login"] else "none (fine: all default models are public)")

r = subprocess.run([sys.executable, "scripts/setup_colab.py", "--profile", PROFILE, "--no-smoke"])
stale = colab.restart_required()
if stale:
    print("Restart the runtime (Runtime > Restart session), then run this cell again:", stale)
assert r.returncode == 0, "setup reported a failure (see the table above)"

In [ ]:
# @title 2. Candidates (backend:voice)
INSTALL_KOKORO = False  # @param {type:"boolean"}
CANDIDATES = ["piper:en_US-libritts_r-medium", "piper:en_US-joe-medium", "piper:en_US-lessac-medium",
              "mms:facebook/mms-tts-eng"]
if INSTALL_KOKORO:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kokoro-onnx>=0.6"], check=True)
    CANDIDATES.append("kokoro:af_heart")
N_SENTENCES = 25  # @param {type:"integer"}

In [ ]:
# @title 3. Run
import time
before = {p.name for p in colab.results_dir().glob("tts-eval*")}
t0 = time.perf_counter()
with colab.GpuMonitor() as mon:
    r = subprocess.run([sys.executable, "scripts/tts_eval.py", "--profile", PROFILE, "--n", str(N_SENTENCES),
                        "--candidates", *CANDIDATES])
new = sorted(p for p in colab.results_dir().glob("tts-eval*") if p.name not in before)
run_dir = new[-1] if new else None
print("exit", r.returncode, "| run", run_dir, "| peak GPU MB", mon.peak_mb, "|", round(time.perf_counter() - t0), "s")

In [ ]:
# @title 4. Table (saved as CSV)
import json, pandas as pd
from IPython.display import display
GPU = (colab.environment_report()["cuda"]["gpus"] or [{"name": "cpu"}])[0]["name"].replace(" ", "_")
rows = []
for x in json.loads((run_dir / "results.json").read_text(encoding="utf-8")):
    if x["status"] != "ok":
        rows.append({"candidate": x["candidate"], "status": "FAILED: " + x.get("error", "")[:80]}); continue
    rows.append({"candidate": x["candidate"], "status": "ok", "license": x["license"],
                 "first_audio_p50_s": round(x["first_audio_ms"]["median"] / 1000, 3),
                 "first_audio_p95_s": round(x["first_audio_ms"]["p95"] / 1000, 3), "rtf": x["rtf"],
                 "roundtrip_wer": x["intelligibility_all"]["wer"],
                 "roundtrip_wer_names_numbers": x["intelligibility_names_numbers"]["wer"],
                 "clips_with_warnings": x["clips_with_warnings"], "rss_delta_mb": x["rss_delta_mb"]})
df = pd.DataFrame(rows); display(df)
csv = colab.results_dir() / f"colab_tts_comparison_{GPU}.csv"; df.to_csv(csv, index=False); print("saved", csv)

In [ ]:
# @title 5. Listen: the same sentence in each voice
from IPython.display import Audio
TEXT = "President Samia Suluhu Hassan arrived in Dodoma on Tuesday; maize prices rose by 12 percent."  # @param {type:"string"}
samples = []
for c in CANDIDATES:
    backend, voice = c.split(":", 1)
    wav = run_dir / f"sample_{backend}_{voice.replace('/', '_')}.wav"
    subprocess.run([sys.executable, "-m", "src.cli", "speak", "--profile", PROFILE, "--backend", backend, "--voice", voice,
                    "--text", TEXT, "--output", str(wav)], capture_output=True)
    if wav.exists():
        print(c); display(Audio(str(wav))); samples.append(wav)
    else:
        print(c, "- synthesis failed")

In [ ]:
RUN_DIRS, EXTRA_FILES, EXPORT_NAME = [run_dir], [csv], "tts-benchmark"
# @title Export: ZIP this notebook's runs (JSON, CSV, audio, logs, images, manifests) and download
import time
zip_path = colab.results_dir() / f"{EXPORT_NAME}-{time.strftime('%Y%m%d-%H%M%S')}.zip"
env_files = sorted((colab.results_dir().parent / "env").glob("*"))[-4:]
z = colab.zip_runs([p for p in RUN_DIRS if p], zip_path, extra_files=EXTRA_FILES + env_files)
print(z)
colab.download(zip_path)